# D5 - Building an Analysis-Ready BIS Dataset

## Objective

In the previous notebooks, we learned how to discover datasets, understand their structure, explore metadata, and retrieve observations from the BIS SDMX API.

In this notebook, we will take the next step: transforming raw BIS data into an analysis-ready dataset.

Rather than focusing on the SDMX protocol, our goal is to prepare the data for downstream analytical tasks such as exploratory data analysis, visualization, forecasting, and machine learning.

By the end of this notebook, we will:

- Retrieve a small BIS dataset
- Convert the SDMX response into a pandas DataFrame
- Assess the quality of the retrieved data
- Prepare the time dimension for analysis
- Decode SDMX codes into human-readable labels
- Produce a clean, analysis-ready dataset

---

## Workflow

```
Retrieve Data
      │
      ▼
Raw SDMX DataFrame
      │
      ▼
Validate & Clean
      │
      ▼
Enrich with Metadata
      │
      ▼
Analysis-Ready Dataset
```

---

## Expected Outcome

By the end of this notebook, we will have a well-structured pandas DataFrame that is ready for visualization, statistical analysis, and machine learning workflows.

In [1]:
# ----------------------------------------------------
# Imports
# ----------------------------------------------------

import sdmx
import pandas as pd

In [2]:
# ----------------------------------------------------
# Connect to the BIS SDMX API
# ----------------------------------------------------

client = sdmx.Client("BIS")

print("Connected to:", client.source.id)
print(client.source.url)

Connected to: BIS
https://stats.bis.org/api/v1


In [3]:
# ----------------------------------------------------
# Retrieve the BIS Total Credit Dataflow
# ----------------------------------------------------

flow_response = client.dataflow()

print(type(flow_response))

print(flow_response.dataflow.keys())

<class 'sdmx.message.StructureMessage'>
dict_keys(['BIS_REL_CAL', 'WS_CBPOL', 'WS_CBS_PUB', 'WS_CBTA', 'WS_CPMI_CASHLESS', 'WS_CPMI_CT1', 'WS_CPMI_CT2', 'WS_CPMI_DEVICES', 'WS_CPMI_INSTITUT', 'WS_CPMI_MACRO', 'WS_CPMI_PARTICIP', 'WS_CPMI_SYSTEMS', 'WS_CPP', 'WS_CREDIT_GAP', 'WS_DEBT_SEC2_PUB', 'WS_DER_OTC_TOV', 'WS_DPP', 'WS_DSR', 'WS_EER', 'WS_GLI', 'WS_LBS_D_PUB', 'WS_LONG_CPI', 'WS_NA_SEC_C3', 'WS_NA_SEC_DSS', 'WS_OTC_DERIV2', 'WS_SPP', 'WS_TC', 'WS_XRU', 'WS_XTD_DERIV'])


In [4]:
# ----------------------------------------------------
# Retrieve the Total Credit Dataflow
# ----------------------------------------------------

flow = flow_response.dataflow["WS_TC"]

print("ID   :", flow.id)
print("Name :", flow.name)

ID   : WS_TC
Name : Total credit


In [5]:
# ----------------------------------------------------
# Retrieve the Data Structure Definition (DSD)
# ----------------------------------------------------

dsd = client.get(resource=flow.structure)

print(type(dsd))

<class 'sdmx.message.StructureMessage'>


In [6]:
# ----------------------------------------------------
# Build a Query
# ----------------------------------------------------

query_key = "Q.IN.P.A.M..A"
print(query_key)

Q.IN.P.A.M..A


In [8]:
import inspect

print(inspect.signature(client.data))

(resource_id: str | None = None, tofile: 'os.PathLike | IO | None' = None, use_cache: bool = False, dry_run: bool = False, **kwargs) -> 'sdmx.message.Message | requests.Request'


In [9]:
help(client.data)

Help on partial in module functools:

functools.partial(<bound method Client.get of <s...object at 0x1552deba0>>, <Resource.data: 'data'>)
    Retrieve SDMX data or metadata with resource_type='data'.

    (Meta)data is retrieved from the :attr:`source` of the current Client. The
    item(s) to retrieve can be specified in one of two ways:

    1. `resource_type`, `resource_id`: These give the type (see :class:`Resource`)
       and, optionally, ID of the item(s). If the `resource_id` is not given, all
       items of the given type are retrieved.
    2. a `resource` object, i.e. a :class:`.MaintainableArtefact`: `resource_type`
       and `resource_id` are determined by the object's class and
       :attr:`id <.IdentifiableArtefact.id>` attribute, respectively.

    Data is retrieved with `resource_type='data'`. In this case, the optional
    keyword argument `key` can be used to constrain the data that is retrieved.
    Examples of the formats for `key`:

    1. ``{'GEO': ['EL', 'ES'

In [11]:
# ----------------------------------------------------
# Retrieve Data
# ----------------------------------------------------

# response = client.data(
#     resource=flow,
#     key=query_key
# )

response = client.data(
    resource_id="WS_TC",
    key=query_key
)

dataset = response.data[0]

print(type(dataset))
print("Series       :", len(dataset.series))
print("Observations :", len(dataset.obs))

xml.Reader got no structure=… argument for StructureSpecificData


<class 'sdmx.model.v21.StructureSpecificDataSet'>
Series       : 3
Observations : 874


In [12]:
# ----------------------------------------------------
# Convert the SDMX Dataset to a pandas DataFrame
# ----------------------------------------------------

rows = []

for series_key, observations in dataset.series.items():

    # Series metadata
    metadata = {
        key: value.value
        for key, value in series_key.values.items()
    }

    # Observations
    for obs in observations:

        rows.append({
            **metadata,
            "TIME_PERIOD": obs.dimension.values["TIME_PERIOD"].value,
            "OBS_VALUE": float(obs.value)
        })

df = pd.DataFrame(rows)

df.head()

,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q1,6.371
1,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q2,6.602
2,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q3,6.482
3,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q4,6.690
4,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1958-Q1,7.095


In [13]:
# ----------------------------------------------------
# Dataset Overview
# ----------------------------------------------------

print(f"Rows    : {len(df):,}")
print(f"Columns : {df.shape[1]}")

display(df.head())

df.info()

Rows    : 874
Columns : 12


,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,FREQ,TITLE_TS,UNIT_MULT,UNIT_MEASURE,TIME_PERIOD,OBS_VALUE
0,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q1,6.371
1,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q2,6.602
2,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q3,6.482
3,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1957-Q4,6.690
4,IN,P,A,M,USD,A,Q,India - Credit to Private non-financial sector...,9,USD,1958-Q1,7.095


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   BORROWERS_CTY  874 non-null    object 
 1   TC_BORROWERS   874 non-null    object 
 2   TC_LENDERS     874 non-null    object 
 3   VALUATION      874 non-null    object 
 4   UNIT_TYPE      874 non-null    object 
 5   TC_ADJUST      874 non-null    object 
 6   FREQ           874 non-null    object 
 7   TITLE_TS       874 non-null    object 
 8   UNIT_MULT      874 non-null    object 
 9   UNIT_MEASURE   874 non-null    object 
 10  TIME_PERIOD    874 non-null    object 
 11  OBS_VALUE      874 non-null    float64
dtypes: float64(1), object(11)
memory usage: 82.1+ KB


In [14]:
# ----------------------------------------------------
# Data Quality Assessment
# ----------------------------------------------------

summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()
})

summary

,Data Type,Missing Values,Unique Values
BORROWERS_CTY,object,0,1
TC_BORROWERS,object,0,1
TC_LENDERS,object,0,1
VALUATION,object,0,1
UNIT_TYPE,object,0,3
TC_ADJUST,object,0,1
FREQ,object,0,1
TITLE_TS,object,0,3
UNIT_MULT,object,0,2
UNIT_MEASURE,object,0,3


In [15]:
# ----------------------------------------------------
# Inspect the Time Dimension
# ----------------------------------------------------

print("Frequency :", df["FREQ"].unique())

print("\nSample TIME_PERIOD values:")

display(
    df["TIME_PERIOD"]
      .drop_duplicates()
      .sort_values()
      .head(10)
)

Frequency : ['Q']

Sample TIME_PERIOD values:


276    1951-Q2
277    1951-Q3
278    1951-Q4
279    1952-Q1
280    1952-Q2
281    1952-Q3
282    1952-Q4
283    1953-Q1
284    1953-Q2
285    1953-Q3
Name: TIME_PERIOD, dtype: object

In [16]:
# ----------------------------------------------------
# Convert to pandas Quarterly Period
# ----------------------------------------------------

df["Quarter"] = pd.PeriodIndex(
    df["TIME_PERIOD"],
    freq="Q"
)

df[["TIME_PERIOD", "Quarter"]].head()

,TIME_PERIOD,Quarter
0,1957-Q1,1957Q1
1,1957-Q2,1957Q2
2,1957-Q3,1957Q3
3,1957-Q4,1957Q4
4,1958-Q1,1958Q1


In [17]:
# ----------------------------------------------------
# Convert to pandas Quarterly Period
# ----------------------------------------------------

df["Quarter"] = pd.PeriodIndex(
    df["TIME_PERIOD"],
    freq="Q"
)

df[["TIME_PERIOD", "Quarter"]].head()

,TIME_PERIOD,Quarter
0,1957-Q1,1957Q1
1,1957-Q2,1957Q2
2,1957-Q3,1957Q3
3,1957-Q4,1957Q4
4,1958-Q1,1958Q1


In [18]:
# ----------------------------------------------------
# Create Calendar Features
# ----------------------------------------------------

df["Year"] = df["Quarter"].dt.year
df["Quarter_Number"] = df["Quarter"].dt.quarter

df[
    [
        "TIME_PERIOD",
        "Quarter",
        "Year",
        "Quarter_Number"
    ]
].head()

,TIME_PERIOD,Quarter,Year,Quarter_Number
0,1957-Q1,1957Q1,1957,1
1,1957-Q2,1957Q2,1957,2
2,1957-Q3,1957Q3,1957,3
3,1957-Q4,1957Q4,1957,4
4,1958-Q1,1958Q1,1958,1


In [19]:
# ----------------------------------------------------
# Reorder Columns
# ----------------------------------------------------

column_order = [

    "Year",
    "Quarter",
    "TIME_PERIOD",

    "BORROWERS_CTY",
    "TC_BORROWERS",
    "TC_LENDERS",

    "VALUATION",
    "UNIT_TYPE",
    "TC_ADJUST",

    "OBS_VALUE"
]

df = df[column_order]

df.head()

,Year,Quarter,TIME_PERIOD,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,OBS_VALUE
0,1957,1957Q1,1957-Q1,IN,P,A,M,USD,A,6.371
1,1957,1957Q2,1957-Q2,IN,P,A,M,USD,A,6.602
2,1957,1957Q3,1957-Q3,IN,P,A,M,USD,A,6.482
3,1957,1957Q4,1957-Q4,IN,P,A,M,USD,A,6.690
4,1958,1958Q1,1958-Q1,IN,P,A,M,USD,A,7.095


In [20]:
# ----------------------------------------------------
# Dataset Summary
# ----------------------------------------------------

summary = pd.DataFrame({

    "Metric": [

        "Observations",
        "Countries",
        "Borrower Sectors",
        "Lender Sectors",
        "Units",
        "Years Covered"

    ],

    "Value": [

        len(df),

        df["BORROWERS_CTY"].nunique(),

        df["TC_BORROWERS"].nunique(),

        df["TC_LENDERS"].nunique(),

        df["UNIT_TYPE"].nunique(),

        f"{df['Year'].min()} - {df['Year'].max()}"

    ]
})

summary

,Metric,Value
0,Observations,874
1,Countries,1
2,Borrower Sectors,1
3,Lender Sectors,1
4,Units,3
5,Years Covered,1951 - 2025


In [21]:
# ----------------------------------------------------
# Retrieve the Data Structure Definition (DSD)
# ----------------------------------------------------

dsd = client.get(resource=flow.structure)

print(type(dsd))

<class 'sdmx.message.StructureMessage'>


In [26]:
# ----------------------------------------------------
# Available Codelists
# ----------------------------------------------------

df_codelists = pd.DataFrame({
    "Codelist ID": list(codelists.keys()),
    "Name": [str(cl.name) for cl in codelists.values()]
})

df_codelists

,Codelist ID,Name
0,CL_ADJUST,Adjustment codelist
1,CL_AREA,Reference area code list
2,CL_BIS_UNIT,BIS_Unit
3,CL_COLLECTION,Collection
4,CL_CONF_STATUS,Observation confidentiality code list
5,CL_DECIMALS,"Decimals codelist (BIS, ECB)"
6,CL_FREQ,Code list for Frequency (FREQ)
7,CL_OBS_STATUS,"Observation status codelist (BIS, ECB, Eurosta..."
8,CL_SUB_CHANNEL,Submission channel
9,P,


In [27]:
# ----------------------------------------------------
# Build a Code Lookup Dictionary
# ----------------------------------------------------

def codelist_to_dict(codelist):

    return {
        code.id: str(code.name)
        for code in codelist.items.values()
    }

In [29]:
# ----------------------------------------------------
# Create Lookup Dictionaries
# ----------------------------------------------------

country_lookup = codelist_to_dict(
    codelists["CL_AREA"]
)

borrower_lookup = codelist_to_dict(
    codelists["CL_TC_BORROWERS"]
)

lender_lookup = codelist_to_dict(
    codelists["CL_TC_LENDERS"]
)

valuation_lookup = codelist_to_dict(
    codelists["CL_VALUATION"]
)

adjust_lookup = codelist_to_dict(
    codelists["CL_ADJUST"]
)

frequency_lookup = codelist_to_dict(
    codelists["CL_FREQ"]
)

In [31]:
# ----------------------------------------------------
# Inspect Available Data Structures
# ----------------------------------------------------

print(type(dsd.structure))
print()

for key in dsd.structure.keys():
    print(key)

<class 'sdmx.dictlike.DictLike'>

BIS_TOTAL_CREDIT


In [32]:
# ----------------------------------------------------
# Retrieve the Data Structure Definition
# ----------------------------------------------------

structure = next(iter(dsd.structure.values()))

print(structure.id)

BIS_TOTAL_CREDIT


In [33]:
# ----------------------------------------------------
# Dimension → Codelist Mapping
# ----------------------------------------------------

mapping = []

for dim in structure.dimensions:

    rep = dim.local_representation

    codelist = None

    if rep is not None and rep.enumerated is not None:
        codelist = rep.enumerated.id

    mapping.append({
        "Dimension": dim.id,
        "Dimension Name": str(dim.concept_identity.name),
        "Codelist": codelist
    })

pd.DataFrame(mapping)

,Dimension,Dimension Name,Codelist
0,FREQ,Frequency,CL_FREQ
1,BORROWERS_CTY,Borrowers' country,CL_AREA
2,TC_BORROWERS,Borrowing sector,CL_TC_BORROWERS
3,TC_LENDERS,Lending sector,CL_TC_LENDERS
4,VALUATION,Valuation method,CL_VALUATION
5,UNIT_TYPE,Unit type,CL_BIS_UNIT
6,TC_ADJUST,Adjustment,CL_ADJUST
7,TIME_PERIOD,Time period or range,None


In [34]:
# ----------------------------------------------------
# Build Lookup Dictionaries
# ----------------------------------------------------

def codelist_to_dict(codelist):
    return {
        code.id: str(code.name)
        for code in codelist.items.values()
    }

lookup_tables = {
    "BORROWERS_CTY": codelist_to_dict(codelists["CL_AREA"]),
    "TC_BORROWERS": codelist_to_dict(codelists["CL_TC_BORROWERS"]),
    "TC_LENDERS": codelist_to_dict(codelists["CL_TC_LENDERS"]),
    "VALUATION": codelist_to_dict(codelists["CL_VALUATION"]),
    "UNIT_TYPE": codelist_to_dict(codelists["CL_BIS_UNIT"]),
    "TC_ADJUST": codelist_to_dict(codelists["CL_ADJUST"]),
    "FREQ": codelist_to_dict(codelists["CL_FREQ"])
}

In [36]:
# ----------------------------------------------------
# Decode SDMX Codes
# ----------------------------------------------------

df_analysis = df.copy()

for column, lookup in lookup_tables.items():

    if column in df_analysis.columns:

        df_analysis[f"{column}_LABEL"] = (
            df_analysis[column]
            .map(lookup)
        )

df_analysis.head()

,Year,Quarter,TIME_PERIOD,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,OBS_VALUE,BORROWERS_CTY_LABEL,TC_BORROWERS_LABEL,TC_LENDERS_LABEL,VALUATION_LABEL,UNIT_TYPE_LABEL,TC_ADJUST_LABEL
0,1957,1957Q1,1957-Q1,IN,P,A,M,USD,A,6.371,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
1,1957,1957Q2,1957-Q2,IN,P,A,M,USD,A,6.602,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
2,1957,1957Q3,1957-Q3,IN,P,A,M,USD,A,6.482,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
3,1957,1957Q4,1957-Q4,IN,P,A,M,USD,A,6.690,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
4,1958,1958Q1,1958-Q1,IN,P,A,M,USD,A,7.095,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks


In [38]:
lookup_tables = {
    "BORROWERS_CTY": codelist_to_dict(codelists["CL_AREA"]),
    "TC_BORROWERS": codelist_to_dict(codelists["CL_TC_BORROWERS"]),
    "TC_LENDERS": codelist_to_dict(codelists["CL_TC_LENDERS"]),
    "VALUATION": codelist_to_dict(codelists["CL_VALUATION"]),
    "UNIT_TYPE": codelist_to_dict(codelists["CL_BIS_UNIT"]),
    "TC_ADJUST": codelist_to_dict(codelists["CL_ADJUST"])
}

In [39]:
# ----------------------------------------------------
# Analysis-Ready Dataset
# ----------------------------------------------------

print(f"Rows    : {len(df_analysis):,}")
print(f"Columns : {df_analysis.shape[1]}")

display(df_analysis.head(10))

Rows    : 874
Columns : 16


,Year,Quarter,TIME_PERIOD,BORROWERS_CTY,TC_BORROWERS,TC_LENDERS,VALUATION,UNIT_TYPE,TC_ADJUST,OBS_VALUE,BORROWERS_CTY_LABEL,TC_BORROWERS_LABEL,TC_LENDERS_LABEL,VALUATION_LABEL,UNIT_TYPE_LABEL,TC_ADJUST_LABEL
0,1957,1957Q1,1957-Q1,IN,P,A,M,USD,A,6.371,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
1,1957,1957Q2,1957-Q2,IN,P,A,M,USD,A,6.602,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
2,1957,1957Q3,1957-Q3,IN,P,A,M,USD,A,6.482,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
3,1957,1957Q4,1957-Q4,IN,P,A,M,USD,A,6.690,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
4,1958,1958Q1,1958-Q1,IN,P,A,M,USD,A,7.095,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
5,1958,1958Q2,1958-Q2,IN,P,A,M,USD,A,7.120,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
6,1958,1958Q3,1958-Q3,IN,P,A,M,USD,A,7.103,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
7,1958,1958Q4,1958-Q4,IN,P,A,M,USD,A,7.327,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
8,1959,1959Q1,1959-Q1,IN,P,A,M,USD,A,7.791,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
9,1959,1959Q2,1959-Q2,IN,P,A,M,USD,A,8.028,India,Private non-financial sector,All sectors,Market value,US dollar,Adjusted for breaks
